# Name
Please write your name down here:

Peter Brederlow

**Supervised learning on mice phenotype data**

Predicting diet from differential expression data was easy with SVMs. It was very neat and regular data, no cells were missing, all values were in a similar range, etc. We will now use a different dataset: the phenotype tables from days 3/4.

In [1]:
from sklearn.experimental import enable_iterative_imputer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, recall_score, matthews_corrcoef, classification_report
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# Data Loading & Preprocessing
Since the phenotype data has a lot of missing values which we don't want to deal with *too* much, we wrote some code to remove samples with a very large number of missing values, and also some features. Our data still has some missing values, but not so much to compromise our models anymore.

In [ ]:
phenotype = pd.read_excel(
     'https://raw.githubusercontent.com/Practical-Integrative-Bioinformatics/Introduction/main/data/phenotype.xlsx',
    #'../../data/phenotype.xlsx',
    sheet_name=None, 
    na_values='x', 
    index_col='@format=column'
)

In [ ]:
# The VO2Max table doesn't follow their own column naming conventions, so fix that first
def rename_vo2_cols(colname):
    return colname.replace('CD', 'CD_').replace('HFD', 'HFD_').replace('__', '_')

phenotype['VO2Max'].rename(columns=rename_vo2_cols, inplace=True)

def split_cd_hfd(input_df):
    input_df.index.name = 'strain'  # change that weird @format=column Excel index label to something meaningful
    input_cd = input_df.filter(regex=r'_CD|CD_') # loc[:, (input_df.columns.str.contains('CD_')) | (input_df.columns.str.contains('_CD'))]
    input_hfd = input_df.filter(regex=r'_HFD|HFD_') #loc[:, (input_df.columns.str.contains('HFD_')) | (input_df.columns.str.contains('_HFD'))]

    input_cd.columns = input_cd.columns.str.replace(r'_CD|CD_', '', regex=True)
    input_hfd.columns = input_hfd.columns.str.replace(r'_HFD|HFD_', '', regex=True)
    
    input_cd.insert(0, 'diet', 'CD')
    input_hfd.insert(0, 'diet', 'HFD')
    
    kept_columns = input_cd.columns.intersection(input_hfd.columns)
    
    df_both = pd.concat([input_cd, input_hfd], sort=False)[kept_columns]
    df_both.columns.name = 'experiment'  # added only later
    
    return df_both.reset_index().set_index(['strain', 'diet']).sort_index()

In [ ]:
pheno = pd.concat([split_cd_hfd(sheet) for sheet_name, sheet in phenotype.items() if sheet_name != "NEW"], axis=1)

# let's remove the main offenders for missing values, both samples and features
pheno = pheno.loc[pheno.T.isna().sum() < 42]  # samples with lots of missing features
pheno = pheno.loc[:, pheno.isna().sum() < 11]  # features with still too many missing values

# a peek at what missing data we are left with:
# sns.clustermap(pheno.isna(), figsize=(10,10))

We will now cut out some features that would make things too easy for our SVM. Bodyweight-associated and food intake features will be banned from our feature list.

In [ ]:
banned = ["KNOWN_BATCH_EFFECT_BY_COHORT_ORDER", "BWGain", "Mass", "BodyWeight", "Food", "Sacrifice"]
kept_cols = [x for x in pheno.columns if all(b not in x for b in banned)]

data_raw = pheno.loc[:, kept_cols]  # unlike the expression table, the phenotype table already has samples as rows so no transposing needed
target = data_raw.index.to_frame()['diet'].replace({'CD': 0, 'HFD': 1})

# let's look at the structure of missing values
sns.clustermap(data_raw.isna())
plt.show()

## Impute missing values

Since most ML algorithms can't deal with NaN values, we will first impute them with the rather primitive `SimpleImputer` class of `sklearn`. Our only choice is whether to use a constant value, the column mean or the column median for the missing values. We will go with median.

Imputer classes have a `fit`, `transform` and `fit_transform` method. Figure out which one you need to fill those missing values.

Unfortunately, SimpleImputer always returns numpy matrices instead of DataFrames, so you should convert the results back to a DF with the correct row/column indices.

In [ ]:
# imputer = SimpleImputer(...)

# data = pd.DataFrame(..., index=data_raw.index, columns=data_raw.columns)

# YOUR CODE HERE
raise NotImplementedError()

## Look at the value ranges of different features. Standardize the data
`data.describe()` can give you an overview of the means, standard deviations, etc. of the different features. If you take a close look at the `mean` or `max` row, you will see that they span 6-7 orders of magnitude, which isn't desirable when using certain machine learning methods, especially linear ones. A PCA would essentially ignore features with small values in favour of features with large values, but SVM's would struggle with them as well.

We should standardize our data so that the scales of different features become comparable. For reasons to be clarified later, we will use the `StandardScaler` class instead of our good old pandas one-liner. It has a similar interface to the previously used `SimpleImputer`: the methods `fit`, `transform` and `fit_transform` are available here as well.

In [ ]:
# scaler = StandardScaler()
# data_std = pd.DataFrame(..., index=data.index, columns=data.columns)

# YOUR CODE HERE
raise NotImplementedError()

## Create a PCA pairplot from the standardized and non-standardized data

Do you agree that feature standardization had a beneficial effect on the PCA transformation? Coloring the data based on diet might help you with your answer.

In [ ]:
# non-standardized: data
# standardized: data_std

# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

# Use an SVC to predict diet from non-standardized phenotype data
Try the `rbf` kernel for a change, and use 3-fold stratified cross-validation to calculate the accuracy. Don't bother with looping over the different training/testing splits manually, use the convencience function `cross_val_score`. It shouldn't take more than 3 short lines of code, or 1 longish line.

Make sure to use the non-standardized `data` DataFrame. Watch out, don't try to run a linear SVM, because it will take a long time to reach convergence and you might have to kill your notebook. Use `kernel='rbf'`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Try the same on the standardized `data_std` DataFrame.
**How does the RBF SVM perform on the standardized data?**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

## Try a linear kernel SVM on the standardized `data_std`
    
Again, just a simple `cross_val_score`. Scikit-learn makes it very easy to test different models with tiny modifications to your code.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Standardize the data fold-by-fold

When we standardized the entire dataset in one go, we cheated a bit. We did not keep the training and testing data fully independent. For a truly honest evaluation, we should derive the standardization parameters (column mean & standard deviation) from the training data only, and apply the same transformation to the test data before prediction. The test data should NOT influence how the training data gets standardized.

You will (temporarily) have to give up on the convenience functions, as you have an extra standardization step to do before fitting and predicting in each fold. Use `StandardScaler`'s `fit_transform` on the training data, and `transform` on the testing data.

**Did it influence the accuracy?**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

## Standardize inside the convenience functions
`sklearn` helps us string together steps by combining them into `Pipeline` objects. We can combine a standardization and a prediction method into a single, indivisible unit, and `sklearn` will make sure that they never violate the principle of keeping the training and testing stages separate, and always uses the appropriate interfaces at every stage.

Those with a keen eye may realize that we cheated the same way when we imputed missing data. You can therefore add the imputation step to the pipeline as well (not that it would change much given how few values had to be imputed).

Once you have created the pipeline, you can pass it to the convenience function `cross_val_score` just like you passed standalone SVC models before.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Take a closer look at the most important features

In the previous notebook we extracted feature weights from linear SVMs by accessing a fitted SVM's `.coef_` attribute. We should do the same now, and take a look at the features that the linear SVM assigned the largest weights to.

Train a `linear` SVM with the entire standardized dataset and extract the features with the 5 highest absolute coefficients. Visualize their values using a `pairplot` and color the samples with their diet.

**What phenotypic traits came out on top? Can you think of an explanation why they would be affected by diet?**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

## Exclude some more features

For the upcoming tasks, we want an imperfect model for demonstration purposes. So we'll cut out the 10 best features to reduce the accuracy of the model. Please keep using `data2` or `data2_std` from now on. You can also experiment with removing even more features.

Quickly check how well an SVM performs on this data.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Sensitivity, specificity, precision...

In some cases, the accuracy of a prediction is secondary to other quality metrics, such as sensitivity or specificity. For example, HIV sceening tests are optimized for sensitivity at the expense of accuracy, ensuring that almost no HIV-positive individuals test negative. This results in an HIV-scare for a lot of HIV-negative individuals each year (as higher sensitivity always implies a higher false positive rate) but in exchange no case of HIV goes undetected on a screening.

We can tune most ML models similarly, and sacrifice accuracy for higher sensitivity or specificity. But first, simply report the sensitivity of your SVM for detecting a high-fat diet (label `1`). You will find tools in `sklearn` that help you calculate this value. Hint: "recall" is a synonym for sensitivity. All the other synonyms and definitions can be found here https://en.wikipedia.org/wiki/Precision_and_recall#Definition_(classification_context)

Make use of the convenience function `cross_val_predict`! It's similar to `cross_val_score`, but instead of returning the accuracies of individual folds, it takes the predicted labels in each fold, and merges + sorts them to match the original order of your samples. This way it guarantees that every prediction comes from a proper cross-validation fold, and the predictions are directly comparable to the target labels.

Sidenote: Feel free to continue with the globally standardized `data2_std` for if you are uncomfortable with the `Pipeline` object.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Make your SVM 95+ percent sensitive for HFD
You can adjust the SVM's parameters to increase your sensitivity for mice on an HFD diet. The `class_weight` keyword of `SVC` takes a dictionary with label-weight pairs (ideally with weights summing to 1).

**How did it affect the overall accuracy, and the false positive rate (i.e. sensitivity for the other, CD condition)? You can create an overview with `classification_report`.**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## ROC curves
We are often interested the relationship between the model's accuracy and sensitivity, or a more commonly used pair of quality measures: false positive rate vs. sensitivity (aka true positive rate). This is what ROC (receiver operating characteristic) curves display: the trade-off between these two qualities.

In the previous task, you used class weights (used during training) to tune sensitivity. An alternative way is "shifting" the decision boundary before producing the output labels.

Most classification ML methods, despite their categorical output, use continuous internal variables for their predictions, and their final decision is a simple thresholding of this continuous variable. For example, in the case of SVMs, this variable is the data point's signed distance from the separating boundary: positive values are assigned to one class, negative values to the other class. Values close to zero (= close to the boundary) are harder to confidently place in either class, and it's down to the arbitrary threshold how they end up being predicted.

You can create a ROC curve by testing how the choice of threshold affects false positive rate and sensitivity. Needless to say, `sklearn` helps you create such plots. All you need to do is extract the SVM's continuous predictive variables, pass it to the appropriate function with the target labels, and plot the results.

Hint: SVC's can return these continuous internal variables if you call `.decision_function(...)` instead of `.predict(...)`. The convenience function `cross_val_predict` gathers the values returned by the classifier's `predict` function by default, but you can change its behaviour with the `method` keyword.

Optional: you can try to plot several ROC curves on top of each other, each resulting from a different shuffled cross-validation run.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

An alternative internal variable: estimated probability. It has to be turned on manually when initializing the `SVC` with `probability=True`, and it will slow things down a bit. The probability values can be accessed by calling `.predict_proba(...)`. In this case you can call `cross_val_predict(..., method='predict_proba')`.

Watch out, `predict_proba` returns two values, the probabilies for either class, summing to one. You are interested in the second value, which is the probability of your `1` label (HFD).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Find the threshold for the desired sensitivity / FPR tradeoff
[You increased sensitivity for HFD](#Make-your-SVM-95+-percent-sensitive-for-HFD) (label 1) by telling the SVM to use a higher weight for that class. Since then, you have learned that you could have instead used the SVM's continuous predictive variables, and threshold them to your own liking, instead of leaving it to the SVM's default (0 for `decision_function` and 0.5 for `predict_proba`).

Your task is to find the threshold value (`decision_function` or `predict_proba`, whichever you prefer) that would achieve 95% sensitivity. Remember, the `roc_curve` function returned three vectors: the ROC plot's FPR values, sensitivity values and the threshold that corresponded to them.

<div class="alert alert-block alert-info"><b>Hint: </b> iterate over the sensitivity and threshold values together, and report the first threshold where sensitivity exceeds 0.95. You can iterate over two lists together using Python's <code>zip</code> function.</div>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()